In [1]:
%load_ext autoreload
%autoreload 2

import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from app.logger import *
import json5,json
import fitz #type: ignore

from app.amc.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

In [40]:
import pandas as pd
import ast

# Load Excel file
df = pd.read_excel(r"tracking_error_22.xlsx")

# Parse each row's string into a dict
df["parsed"] = df["data"].apply(ast.literal_eval)

# Normalize the dicts into columns
df2 = pd.json_normalize(df["parsed"])

# Select only the columns you want
df2 = df2[["Scheme_Name", "Benchmark", "RegularPercent", "DirectPercent", "mfId","date"]]

print(df2.head())

                       Scheme_Name               Benchmark  RegularPercent  \
0  HDFC Gold Exchange Traded Fund.  Domestic Price of Gold            0.28   
1  HDFC Gold Exchange Traded Fund.  Domestic Price of Gold            0.28   
2  HDFC Gold Exchange Traded Fund.  Domestic Price of Gold            0.28   
3  HDFC Gold Exchange Traded Fund.  Domestic Price of Gold            0.28   
4  HDFC Gold Exchange Traded Fund.  Domestic Price of Gold            0.27   

   DirectPercent mfId         date  
0           0.28    9  01-jun-2022  
1           0.28    9  02-jun-2022  
2           0.28    9  03-jun-2022  
3           0.28    9  06-jun-2022  
4           0.27    9  07-jun-2022  


In [41]:
df2.to_csv("2022_tracking_error_amfii.csv")

In [ ]:
import pandas as pd
import os

path = r"C:\Users\kaustubh.keny\Downloads\EQD"
all_dfs = []

for file in os.listdir(path):
    if file.endswith((".xlsx", ".xls")):
        xls_path = os.path.join(path, file)
        try:
            # Try first sheet name
            try:
                df = pd.read_excel(xls_path, sheet_name="Category wise data", header=1)
            except ValueError:
                # Fall back to alternative sheet name
                df = pd.read_excel(xls_path, sheet_name="Category Data", header=1)

            # Filter rows where Month is not null
            df_months = df[df["Month"].notna()]
            all_dfs.append(df_months)

        except Exception as e:
            print(f"Skipping {file}: {e}")

# Concatenate all sheets
if all_dfs:
    consolidated = pd.concat(all_dfs, ignore_index=True)
    output_path = os.path.join(path, "consolidated_all_data.xlsx")
    consolidated.to_excel(output_path, index=False)
    print(f"Consolidated file saved to {output_path}")

In [ ]:
import pandas as pd
import os

path = r"C:\Users\kaustubh.keny\Downloads\EQD"

all_dfs = []

for file in os.listdir(path):
    if file.endswith((".xlsx", ".xls")):
        xls_path = os.path.join(path, file)
        try:
            df = pd.read_excel(xls_path, sheet_name="Category wise data", header=1)
            # keep only rows where Month column is not null/empty
            df_months = df[df["Month"].notna()]
            # add source file info if useful
            df_months["SourceFile"] = file
            all_dfs.append(df_months)
        except Exception as e:
            print(f"Skipping {file}: {e}")

# concatenate all filtered dataframes
if all_dfs:
    consolidated = pd.concat(all_dfs, ignore_index=True)
    print(consolidated.head())
    # save to Excel if needed
    consolidated.to_excel(os.path.join(path, "consolidated_all_months.xlsx"), index=False)

In [ ]:
import requests, time
import pandas as pd
from datetime import datetime, timedelta

def fetch_tracking_error(from_date: str, to_date: str, mf_id: str = "all", output_file: str = "tracking_error.xlsx"):
    """
    Fetch AMFI tracking error data for each day in range and save to one sheet.
    
    Args:
        from_date (str): Start date in format 'dd-mon-yyyy' (e.g. '25-nov-2025')
        to_date (str): End date in format 'dd-mon-yyyy'
        mf_id (str): Mutual Fund ID parameter (default 'all')
        output_file (str): Output Excel file
    """
    # Parse dates
    start = datetime.strptime(from_date, "%d-%b-%Y")
    end = datetime.strptime(to_date, "%d-%b-%Y")

    all_data = []

    # Loop day by day
    current = start
    while current <= end:
        strdt = current.strftime("%d-%b-%Y").lower()  # API expects lowercase month
        url = f"https://www.amfiindia.com/api/tracking-error-data?MF_ID={mf_id}&strdt={strdt}"
        print(f"Fetching {url} ...")

        try:
            resp = requests.get(url, verify=False)
            resp.raise_for_status()
            data = resp.json()  # API returns JSON

            # Flatten into DataFrame
            df = pd.DataFrame(data)

            # Add extra columns
            df["date"] = strdt
            df["MF_ID"] = mf_id

            all_data.append(df)
            time.sleep(5)

        except Exception as e:
            print(f"Failed for {strdt}: {e}")

        current += timedelta(days=1)

    # Combine all
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        final_df.to_excel(output_file, index=False)
        print(f"Saved {len(final_df)} rows to {output_file}")
    else:
        print("No data fetched.")

# Example usage
fetch_tracking_error("01-jun-2022", "31-dec-2022", mf_id="all", output_file="tracking_error_22.xlsx")

In [19]:
import mysql.connector
from mysql.connector import Error
 
def json_to_db(json_path):
    
    file_name = json_path.split("\\")[-1]
    json_string = ""
    with open(json_path,"r") as f:
        data = json.load(f)
        json_string = json.dumps(data) 
    
    print(file_name)
    print(json_string[:10])
    
    flag = False
   
    sp_list = {
        "kim":"mf_processjson_kim",
        "sid":"mf_processjson_sid",
        "fs":"mf_processjson_factsheet"
    }
   
    db_config = {
        "host":"172.22.225.155",
        "port":3306,
        "user":"cog_mf",
        "password":"bnYwFChjLAV2Z%9E",
        "database":"cog_mf"
    }
    try:
        sp_name = None
   
        if "_kim" in file_name.lower(): sp_name = sp_list.get("kim","")
        elif "_sid" in file_name.lower(): sp_name = sp_list.get("sid","")
        elif "_fs.json" in file_name.lower(): sp_name = sp_list.get("fs","")
 
        print(sp_name)
        conn = mysql.connector.connect(**db_config)
        cursor = conn.cursor()
        
        print(sp_name)
        print("DB is connected.")
 
        try:
            if "_FS.json" in file_name:
                cursor.callproc("mf_update_document_details_FS", [file_name])
                print("First SP run.")
 
            cursor.callproc(sp_name, [json_string])
            conn.commit()
            print("Second SP run.")
            flag = True
            
        except Error as err:
            
            conn.rollback() #rollback -> if error
            print("Rolled Back [ERROR]:", err)

        finally:
            cursor.close()
            conn.close()
            print("Connection closed.")

    except Exception as err:
        #something
        print("Error in db connection.")
    
    return flag


path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\38_31-Oct-25_FS.json"
json_to_db(path)

38_31-Oct-25_FS.json
{"metadata
mf_processjson_factsheet
mf_processjson_factsheet
DB is connected.
First SP run.
Second SP run.
Connection closed.


True

In [ ]:
path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\NSE_NEWS_PROGRAM\List of Companies.xlsx"
df1 = pd.read_excel(path)
df1.head(10)

In [ ]:
path = r"C:\Users\kaustubh.keny\Downloads\Company Name and Ticker List.xlsx"
import pandas as pd

df = pd.read_excel(path)
df.head(10)

In [15]:

import mysql.connector
from mysql.connector import Error

def json_to_db(json_path):
    
    file_name = json_path.split("\\")[-1]
    json_string = ""
    with open(json_path,"r") as f:
        json_string = str(json.load(f))
    
    print(file_name)
    print(json_string[:10])
    
    flag = False
    
    sp_list = {
        "kim":"mf_processjson_kim",
        "sid":"mf_processjson_sid",
        "fs":"mf_processjson_factsheet"
    }
    
    db_config = {
        "host":"172.22.225.155",
        "port":3306,
        "user":"cog_mf",
        "password":"bnYwFChjLAV2Z%9E",
        "database":"cog_mf"
    }
    print(db_config)
    try:
        sp_name = None
    
        if "_kim" in file_name.lower(): sp_name = sp_list.get("kim","")
        elif "_sid" in file_name.lower(): sp_name = sp_list.get("sid","")
        elif "_fs.json" in file_name.lower(): sp_name = sp_list.get("fs","")

        
        print(sp_name)
        
        conn = mysql.connector.connect(**db_config)
        cursor = conn.cursor()
        print("DB Connected.")

        # try:
        #     if "_FS.json" in file_name:
        #         cursor.callproc("mf_update_document_details_FS", [file_name])

        #     cursor.callproc(sp_name, [json_string])
        #     conn.commit()
        #     flag = True
        # except Error as err:
        #     conn.rollback()

        # finally:
        #     cursor.close()
        #     conn.close()
    except Exception as err:
        #something
        print("error in connecting db")
        print(err)

    # return flag


path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\6_31-Oct-25_FS.json"

json_to_db(path)

6_31-Oct-25_FS.json
{'metadata
{'host': '172.22.225.155', 'port': 3306, 'user': 'cog_mf', 'password': 'bnYwFChjLAV2Z%9E', 'database': 'cog_mf'}
mf_processjson_factsheet
DB Connected.


In [36]:
import requests

url = "https://api.mospi.gov.in/api/plfs/getData"
params = {
    "indicator_code": 1,
    "page": 1,
    "limit": 20,
    "year": "2023-24",
    "age_code": 2,
    "education_code": 4,
    "gender_code": 1,
    "religion_code": 1,
    "sector_code": "1,2,3",
    "social_category_code": "1,2,3,4,5",
    "state_code": "1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,99",
    "weekly_status_code": "1,2"
}

resp = requests.get(url, params=params, timeout=30)
print(resp.status_code)
print(resp.json())

200
{'data': [], 'msg': 'No Data Found', 'statusCode': True}


In [ ]:
from selenium import webdriver
import requests

driver = webdriver.Chrome()
driver.get("https://www.mca.gov.in/")

# Perform a search manually or via Selenium
cookies = driver.get_cookies()

session = requests.Session()
for cookie in cookies:
    session.cookies.set(cookie['name'], cookie['value'])

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.mca.gov.in/"
}

url = "https://www.mca.gov.in/bin/mca/mds/commonSearch"
params = {
    "module": "MDS",
    "searchKeyWord": "INFOSYS",
    "searchType": "autosuggest",
    "mdsSearchType": "company"
}

resp = session.get(url, params=params, headers=headers, timeout=20, verify=True)
print(resp.status_code, resp.text)

In [ ]:
path = r"C:\Users\kaustubh.keny\Downloads\AMC"
import re
from datetime import datetime


def file_details(file_name: str):
    get_name = "(\\d{2,3}_\\d{2}-[A-Za-z]{3}-\\d{2}(?:_\\d{1})?)_FS.pdf"
    if matches:= re.findall(get_name, file_name):
        content = matches[0].split("_")
        code,dateval,*rest = content
        date_obj = datetime.strptime(dateval, "%d-%b-%y")
        
        if rest:
            final_code = code + "_1"
        else:
            final_code = code + "_0"
        return final_code,str(date_obj.year)
    return None,None

def load_config(code,year):
    base_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config"
    config_path = os.path.join(base_path, year, f"{code}_AMC.json5")
    print(config_path)
    if not os.path.exists(config_path):
        return None
    with open(config_path, "r") as f:
        config = json5.load(f)
    
    return config

print(load_config("1_0","2025"))

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config\2025\1_0_AMC.json5
{'AMC_NAME': 'Axis Mutual Fund', 'MAIN_MAP': {'select': True, 'map': True, 'special': True}, 'IMP_DATA': {'amc_name': 'Axis Asset Management Company Limited', 'mutual_fund_name': 'Axis Mutual Fund'}, 'PARAMS': {'title': {'ocr': False, 'pattern': '(AXIS.*?(?:FUNDS?|ETF[Ss]?|INDEX|PLAN|PATH|F[Oo]F)\\s*(?:OF\\s*FUNDS?|F[Oo]F|FUNDS?|FUND OF FUNDS?|.+?PLAN)?)', 'bbox': [0, 0, 580, 36]}, 'clip_bbox': [[0, 90, 200, 500], [180, 90, 360, 500], [0, 470, 360, 820]], 'line_x': 260.0, 'data': {'size': [6, 12], 'color': [23], 'update_size': 30.0, 'font': ['Lato-Bold']}, 'content_size': [30.0, 10.0], 'method': 'clip', 'line_side': 'both', 'max_financial_index_highlight': 7, 'pdf_ocr': False, 'sanitize_fund': False, 'stop_words': []}, 'DUPLICATE_MUTUAL_FUNDS': {}, 'PRE_DATA_SELECT': [], 'REGEX': {'date': '(\\d{1,2}\\s*(?:th|rd|st|nd)\\s*(?:Jan|Feb|Mar|Apr|May|June|July|Aug|Sep|Oct|Nov|Dec)\\D*\\d{4})', 'benchmark': '

In [42]:
params = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config\2019\param2019.json5"
year_path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config\2019"
import json5

with open(params, "r") as f:
    parameters = json5.load(f)
    for key, value in parameters.items():
        # print(f"{key}_AMC.json5")
        file_name = f"{key}_AMC.json5"
        
        path = os.path.join(year_path,file_name)
        with open(path,"w") as f:
            json5.dump(value,f)

In [ ]:
"""360 ONE FILE CODE"""
amc_id = '18_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\360 ONE Mutual Fund\18_31-Dec-21_FS.pdf"
object = ThreeSixtyOne(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [ ]:
with open("data.json","w+") as file:
    json.dump(data,file)
with open("extract.json","w+") as file:
    json.dump(dfs,file)
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

In [ ]:
"""ANGEL ONE FILE CODE"""
amc_id = '96_0'
# logging.info(f"User Ran Fund Data of {amc_name}")
 path= mutual_fund[amc_id]
object = AngelOne(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text,map_keys = True, special_handling = True)
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

In [ ]:
"""SHRIRAM MAIN FILE CODE"""
amc_id = '36_0'
# logging.info(f"User Ran Fund Data of {amc_name}")
 path= mutual_fund[amc_id]
object = Shriram(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text,map_keys = True, special_handling = True)
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

In [ ]:
'''AXIS ONE ACTIVE MAIN FILE CODE''' #No issues as of now
amc_name = "Axis Mutual Fund"
logger.info(f"User Ran Fund Data of {amc_name}")
path = mutual_fund[amc_name]
object = AXISMF(amc_name,path)

title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)


In [ ]:
object = AXISMF(amc_name,path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
Helper.quick_json_dump(extracted_text,object.JSONPATH)

In [ ]:
object = AXISMF(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text,select = True, map_keys = True, special=True)
Helper.quick_json_dump(dfs,object.JSONPATH)

In [5]:
"""ADITYA BIRLA FILE CODE"""# pages = [17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49, 50, 52, 54, 56, 58, 60, 62, 64, 66, 67, 69, 72, 75, 78, 80, 82, 86, 87, 89, 91, 93, 95, 98, 101, 103, 106, 108, 110, 112, 114, 115, 116, 118, 119, 121, 123, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 143, 145, 147, 149, 151, 152, 153, 155, 157, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182]
amc_id = '3_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Aditya Birla Sun Life Mutual Fund\3_31-Dec-21_FS.pdf"
object = AdityaBirla(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [7]:
object = AdityaBirla(amc_id,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [10]:
"""BANDHAN MF FILE MAIN CODE""" # rgex for fund manager

amc_name = "16_0"
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Bandhan Mutual Fund\16_31-Dec-21_FS.pdf"

object = Bandhan(amc_name,path)
title,path_pdf= object.check_and_highlight(path)
data  = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [12]:
object = Bandhan(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [13]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\16_31-Dec-21_FS.json


In [ ]:
""" BARODA BNP MAIN FILE CODE"""

amc_name = 'Baroda Bnp Paribas Mutual Fund'
logging.info(f"User Ran Fund Data of {amc_name}")
path = mutual_fund[amc_name]
object = BarodaBNP(amc_name,path)
title,path_pdf= object.check_and_highlight(path)

In [ ]:
data  = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [ ]:
object = BarodaBNP(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text, select = True, map_keys = True, special = True)
Helper.quick_json_dump(dfs, object.JSONPATH)

In [15]:
"""BANK OF INDIA"""

amc_id = '5_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Bank of India Mutual Fund\5_31-Dec-21_FS.pdf"
object = BankOfIndia(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)


In [16]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\5_31-Dec-21_FS.json


In [ ]:
"""BAJAJ FINSERV MAIN FILE CODE""" 

amc_id = "59_0"
# logging.info(f"User Ran Fund Data of {amc_name}")
path= mutual_fund[amc_id]
object = BajajFinServ(amc_id,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data,is_table= object.MAIN_MAP["table"])
object = BajajFinServ(amc_id,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)
# Helper.quick_json_dump(dfs, object.JSONPATH)

In [2]:
"""CANARA MUTUAL FUND"""

amc_id = '6_0'
path = r"C:\Users\kaustubh.keny\Downloads\6_31-Oct-25_FS.pdf"
object = Canara(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [7]:
object =  Canara(amc_id,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [8]:
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\6_31-Oct-25_FS.json


In [ ]:
"""DSP MAIN FILE CODE"""

amc_id = '8_0'
# logging.info(f"User Ran Fund Data of {amc_name}")
 path= mutual_fund[amc_id]
object = DSP(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)

In [ ]:
# object = DSP(amc_id,path)
extracted_text = object.get_generated_content(data, object.MAIN_MAP["table"])
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text,map_keys = True,special_handling = True)
with open("dfs.json","w+") as file:
    json.dump(dfs,file)

In [22]:
"""EDELWEISS MP FILE MAIN CODE"""

amc_id = '9_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Edelweiss Mutual Fund\9_31-Dec-21_FS.pdf"
object = Edelweiss(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)


In [39]:
object = Edelweiss(amc_id,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [40]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\9_31-Dec-21_FS.json


In [42]:
"""FRANKLIN TEMPLETON FILE MAIN CODE"""

amc_name = "11_0"
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Franklin Templeton Mutual Fund\11_31-Dec-21_FS.pdf"
object = FranklinTempleton(amc_name,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)


In [44]:
object = FranklinTempleton(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)



In [45]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\11_31-Dec-21_FS.json


In [ ]:
"""GROWW MUTUAL FUND MAIN CODE"""# pages = [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26, 28, 29, 30, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42]

amc_id = '20_0'
# logging.info(f"User Ran Fund Data of {amc_name}")
 path= mutual_fund[amc_id]
object = GROWW(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [ ]:
object = GROWW(amc_id,path)
final_text = object.refine_extracted_data(extracted_text)
# dfs = object.merge_and_select_data(final_text)
# Helper.quick_json_dump(final_text,object.JSONPATH)
with open("test.json","w+") as file:
    json.dump(final_text,file)

Function Running: refine_extracted_data


In [16]:
"""HDFC MUTUAL FUND""" 
amc_name = "12_0"
path = r"C:\Users\kaustubh.keny\Downloads\12_31-Oct-25_FS.pdf"
object = HDFC(amc_name,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [23]:
object = HDFC(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [26]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\12_31-Oct-25_FS.json


In [ ]:
"""HDFC MUTUAL FUND PASSIVE""" # pages = [5, 6, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

amc_name = "Hdfc Mutual Fund Passive"
logging.info(f"User Ran Fund Data of {amc_name}")
path = mutual_fund[amc_name]
object = HDFC(amc_name,path)
title,path_pdf= object.check_and_highlight(path)

data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text,flatten = True)
dfs = object.merge_and_select_data(final_text, select = True,map_keys = True,special=True)
Helper.quick_json_dump(dfs,object.JSONPATH)

In [ ]:
"""HELIOS MF FILE MAIN CODE""" # pages = [2, 4, 6, 8,10]

amc_name = 'Helios Mutual Fund'
logging.info(f"User Ran Fund Data of {amc_name}")
path = mutual_fund[amc_name]
object = Helios(amc_name,path)
title,path_pdf= object.check_and_highlight(path)

data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)
Helper.quick_json_dump(dfs,object.JSONPATH)

In [ ]:
""" ICICI MF MAIN FILE CODE""" #pages = [8, 9, 11, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 46, 47, 49, 51, 53, 55, 56, 58, 59, 60, 62, 64, 65, 66, 67, 68, 69, 71, 72, 74, 75, 77, 78, 80]

amc_id = '14_0'
# logging.info(f"User Ran Fund Data of {amc_name}")
 path= mutual_fund[amc_id]
object = ThreeSixtyOne(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
# dfs = object.merge_and_select_data(final_text)
# Helper.quick_json_dump(dfs, object.JSONPATH)

In [53]:
""" ICICI MF MAIN FILE CODE PASSIVE"""
amc_name = "14_0"
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\ICICI Prudential Mutual Fund\14_31-Dec-21_FS.pdf"
object = ICICI(amc_name,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [55]:
object = ICICI(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)


In [5]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\14_31-Oct-25_1_FS.json


In [3]:
""" ICICI MF MAIN FILE CODE PASSIVE"""
amc_name = "14_1"
path = r"C:\Users\kaustubh.keny\Downloads\AMC\14_31-Oct-25_1_FS.pdf"
object = ICICIPassive(amc_name,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [29]:
object = ICICIPassive(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)


In [30]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\14_31-Oct-25_1_FS.json


In [57]:
""" INVESCO MF MAIN FILE CODE""" # pages = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43]

amc_name = '21_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Invesco Mutual Fund\21_31-Dec-21_FS.pdf"
object = Invesco(amc_name,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [58]:
object = Invesco(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [5]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\14_31-Oct-25_1_FS.json


In [60]:
"""ITI MAIN FILE CODE""" 
amc_name = '51_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\ITI Mutual Fund\51_31-Dec-21_FS.pdf"
object = ITI(amc_name,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [69]:
object = ITI(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs= object.merge_and_select_data(final_text)

In [70]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\51_31-Dec-21_FS.json


In [71]:
"""JM FUND MAIN FILE CODE"""

amc_id = '22_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\JM Financial Mutual Fund\22_30-Sep-21_FS.pdf"
object = JMMF(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [74]:
object = JMMF(amc_id,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [75]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\22_30-Sep-21_FS.json


In [3]:
"""KOTAK FUND MAIN CODE"""
amc_id = '23_0'
path = r"C:\Users\kaustubh.keny\Downloads\23_31-Oct-25_FS.pdf"
object = Kotak(amc_id,path)
title,path_pdf= object.check_and_highlight(path)

data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)


In [4]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\23_31-Oct-25_FS.json


In [ ]:
"""LIC FUND MAIN CODE""" #issue: OCR for title bcz they're images
amc_id = '25_0'
path = r"C:\Users\kaustubh.keny\Downloads\AMC\25_31-Oct-25_FS.pdf"

config = get_config("2025",amc_id)
regex = get_regex("2025")


object = LIC(config,regex,path)
# print(object.__init__)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [17]:
config = get_config("2025",amc_id)
regex = get_regex("2025")
object = LIC(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)
# Helper.quick_json_dump(dfs, object.JSONPATH)

In [18]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\25_31-Oct-25_FS.json


In [8]:
with open("test.json","w+") as file:
    json.dump(dfs,file)

In [8]:
"""MAHINDRA MANULIFE MAIN CODE""" #incrase pages count

amc_name = '26_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Mahindra Manulife Mutual Fund\26_31-Dec-21_FS.pdf"
object = MahindraManu(amc_name,path)
title,path_pdf = object.check_and_highlight(path)

data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
# object = MahindraManu(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [9]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\26_31-Dec-21_FS.json


In [ ]:
"""MOTILAL OSWAL PASSIVE MAIN CODE FILE""" #Fund Manager Regex
amc_id = '28_1'
# logging.info(f"User Ran Fund Data of {amc_name}")
 path= mutual_fund[amc_id]
object = MotilalOswalPassive(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
# dfs = object.merge_and_select_data(final_text)
# Helper.quick_json_dump(dfs, object.JSONPATH)

In [19]:
"""MOTILAL OSWAL MAIN CODE FILE"""

amc_id = '28_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Motilal Oswal Mutual Fund\28_31-Dec-21_FS.pdf"
object = MotilalOswal(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [20]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\28_31-Dec-21_FS.json


In [ ]:
"""MIRAE MAIN FILE CODE PASSIVE"""

amc_id = '27_1'
object = MIRAE(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
# dfs = object.merge_and_select_data(final_text)
# Helper.quick_json_dump(dfs, object.JSONPATH)

In [17]:
"""MIRAE MAIN FILE CODE"""
amc_id = '27_0'
path= r"C:\Users\kaustubh.keny\Downloads\2021_changed\Mirae Asset Mutual Fund\27_31-Dec-21_FS.pdf"
object = MIRAE(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)


In [18]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\27_31-Dec-21_FS.json


In [ ]:
object = NJMF(amc_name,path)
final_text = object.refine_extracted_data(extracted_text,flatten = True)
dfs = object.merge_and_select_data(final_text)
Helper.quick_json_dump(dfs, object.JSONPATH)

In [27]:
"""PGIM MAIN FILE CODE""" #ISSUES: Clean data further

amc_id = '7_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\PGIM India Mutual Fund\7_31-Dec-21_FS.pdf"
object = PGIM(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [28]:

save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\7_31-Dec-21_FS.json


In [5]:
"""QUANT MAIN CODE FILE"""

amc_id = '10_0'
path = r"C:\Users\kaustubh.keny\Downloads\10_31-Oct-25_FS.pdf"
object = QuantMF(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [6]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\10_31-Oct-25_FS.json


In [25]:
"""THE WEALTH COMPANY""" 

amc_id = '100_0'
path = r"C:\Users\kaustubh.keny\Downloads\100_31-Oct-25_FS.pdf"
object = WealthCompany(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [ ]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

In [2]:
"""QUANTUM MAIN FILE CODE""" 

amc_id = '32_0'
path = r"32_31-Oct-25_FS.pdf"
object = Quantum(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [3]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

with open("data.json", 'w') as f:
  json.dump(final_text, f, indent=2)
  
with open("extract.json", 'w') as f:
  json.dump(extracted_text, f, indent=2)

with open(save_path, 'r') as f:
  data = json.load(f)


File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\32_31-Oct-25_FS.json


In [23]:
"""SAMCO PDF FILE MAIN CODE"""

amc_name = '58_0'
path =r"C:\Users\kaustubh.keny\Downloads\58_31-Oct-25_FS.pdf"
object = Samco(amc_name,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [31]:
object = Samco(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [32]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

with open("data.json", 'w') as f:
  json.dump(final_text, f, indent=2)
  
with open("extract.json", 'w') as f:
  json.dump(extracted_text, f, indent=2)

with open(save_path, 'r') as f:
  data = json.load(f)


File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\58_31-Oct-25_FS.json


In [8]:
"""SBI PASSIVE PDF FILE MAIN CODE""" 

amc_id = '35_1'

config = get_config("2025",amc_id)
regex = get_regex("2025")

path = r"C:\Users\kaustubh.keny\Downloads\AMC\35_31-Oct-25_1_FS.pdf"
object = SBIPassive(config,regex,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [17]:
config = get_config("2025",amc_id)
regex = get_regex("2025")
object = SBIPassive(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [18]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

with open("data.json", 'w') as f:
  json.dump(final_text, f, indent=2)
  
with open("extract.json", 'w') as f:
  json.dump(extracted_text, f, indent=2)

with open(save_path, 'r') as f:
  data = json.load(f)


File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\35_31-Oct-25_1_FS.json


In [ ]:
pattern = r'(?:Mr|Mrs|Ms)\.?\s*([A-Za-z]+\s+[A-Za-z]+).+Fund\s+Since\s*([A-Za-z]+\s*\d{4}).+?Over\s*(\d{2}\s*years)'
for fund, content in final_text.items():
    check = 'before.fund_managers'
    if check in content:
        text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[check]).strip()
        print(text)
        match = re.findall(pattern,text, re.IGNORECASE)
        print(match)

In [ ]:
"""SBI PDF FILE MAIN CODE""" 
amc_id = '35_0'
# logging.info(f"User Ran Fund Data of {amc_name}")
 path= mutual_fund[amc_id]
object = SBIPassive(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
# dfs = object.merge_and_select_data(final_text,select = True,map_keys = True,special = True)
# Helper.quick_json_dump(dfs,object.JSONPATH)

In [ ]:
"""SUNDARAM MAIN FILE CODE"""

amc_name = 'Sundaram Mutual Fund'
logging.info(f"User Ran Fund Data of {amc_name}")
path = mutual_fund[amc_name]
object = Sundaram(amc_name,path)
title,path_pdf = object.check_and_highlight(path)

data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [13]:
Helper.quick_json_dump(extracted_text,object.JSONPATH)


  JSON saved at C:\Users\rando\OneDrive\Documents\mywork-repo\data\output\dump_sundaram_21_43_41.json


In [ ]:
object = Sundaram(amc_name,path)
final_text = object.refine_extracted_data(extracted_text,flatten = True)
dfs = object.merge_and_select_data(final_text,select = True, map_keys = True, special=True)
Helper.quick_json_dump(dfs,object.JSONPATH)

Function Running: refine_extracted_data
Function Running: merge_and_select_data

  JSON saved at C:\Users\rando\OneDrive\Documents\mywork-repo\data\output\dump_sundaram_21_46_34.json


In [20]:
"""TATA FILE MAIN CODE """
amc_id = '38_0'
path = r"C:\Users\kaustubh.keny\Downloads\38_31-Oct-25_FS.pdf"
config = get_config("2025",amc_id)
regex = get_regex("2025")
object = Tata(config,regex,path)

title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)


In [23]:
config = get_config("2025",amc_id)
regex = get_regex("2025")
object = Tata(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [24]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

with open("data.json", 'w') as f:
  json.dump(final_text, f, indent=2)
  
with open("extract.json", 'w') as f:
  json.dump(extracted_text, f, indent=2)

with open(save_path, 'r') as f:
  data = json.load(f)


File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\38_31-Oct-25_FS.json


In [24]:
"""TAURUS MAIN FILE CODE"""
amc_name = '39_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Taurus Mutual Fund\39_31-Dec-21_FS.pdf"
object = Taurus(amc_name,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [25]:
object = Taurus(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [27]:
"""TRUST MAIN FILE CODE"""
amc_name = '55_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Trust Mutual Fund\55_31-Dec-21_FS.pdf"
object = Trust(amc_name,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [36]:
object = Trust(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [37]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

with open("data.json", 'w') as f:
  json.dump(final_text, f, indent=2)
  
with open("extract.json", 'w') as f:
  json.dump(extracted_text, f, indent=2)

with open(save_path, 'r') as f:
  data = json.load(f)


File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\55_31-Dec-21_FS.json


In [4]:
"""UTI MAIN FILE CODE"""

amc_id = '41_0'
path = r"41_31-Oct-25_FS.pdf"
object = UTI(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [2]:
"""UTI MAIN FILE CODE PASSIVE"""

amc_id = '41_1'
path = r"41_31-Oct-25_1_FS.pdf"
object = UTIPassive(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [5]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

with open("data.json", 'w') as f:
  json.dump(final_text, f, indent=2)
  
with open("extract.json", 'w') as f:
  json.dump(extracted_text, f, indent=2)

with open(save_path, 'r') as f:
  data = json.load(f)


File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\41_31-Oct-25_FS.json


In [38]:
"""UNION MUTUAL FUND""" #Issues: Major Issue

amc_name = '40_0'
path = r"C:\Users\kaustubh.keny\Downloads\2021_changed\Union Mutual Fund\40_31-Dec-21_FS.pdf"
object = Union(amc_name,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [ ]:
object = Union(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [ ]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

with open("data.json", 'w') as f:
  json.dump(final_text, f, indent=2)
  
with open("extract.json", 'w') as f:
  json.dump(extracted_text, f, indent=2)

with open(save_path, 'r') as f:
  data = json.load(f)


In [ ]:
"""Unifi MUTUAL FUND""" #Issues: Major Issue

amc_id = '97_0'
 path= mutual_fund[amc_id]
object = Unifi(amc_id,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text,map_keys = True, special_handling = True)
# Helper.quick_json_dump(dfs, object.JSONPATH)

In [ ]:
"""WHITEOAK MUTUAL FUND"""

amc_name = 'Whiteoak Mutual Fund'
logging.info(f"User Ran Fund Data of {amc_name}")
path = mutual_fund[amc_name]
object = WhiteOak(amc_name,path)
title,path_pdf= object.check_and_highlight(path)
# pages = [7, 9, 11, 13, 15, 17, 19, 20, 21, 23, 25, 26, 28, 30, 32, 34, 36, 37]
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [ ]:
object = WhiteOak(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
# dfs = object.merge_and_select_data(final_text)
# Helper.quick_json_dump(dfs,object.JSONPATH)

Function Running: refine_extracted_data


In [ ]:
"""ZERODHA MAIN CODE""" 

amc_name = '71_0'
path =r"C:\Users\kaustubh.keny\Downloads\71_30-Sep-25_FS.pdf"
object = Zerodha(amc_name,path)
title,path_pdf= object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [7]:
object = Zerodha(amc_name,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

In [4]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
print(f"File Saved At: {save_path}")

with open("data.json", 'w') as f:
  json.dump(final_text, f, indent=2)
  
with open("extract.json", 'w') as f:
  json.dump(extracted_text, f, indent=2)

with open(save_path, 'r') as f:
  data = json.load(f)


File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\71_30-Sep-25_FS.json


In [16]:
""""HSBC MAIN FILE CODE"""

amc_id = '13_0'
path = r"C:\Users\kaustubh.keny\Downloads\AMC\13_31-Oct-25_FS.pdf"

config = get_config("2025",amc_id)
regex = get_regex("2025")
object = HSBC(config,regex,path)
title,path_pdf = object.check_and_highlight(path)
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)


In [31]:
config = get_config("2025",amc_id)
regex = get_regex("2025")
object = HSBC(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

[HSBC._special_match_regex_to_content] (13_31-Oct-25_FS.pdf) TypeError: expected string or bytes-like object, got 'NoneType'
[HSBC._special_match_regex_to_content] (13_31-Oct-25_FS.pdf) TypeError: expected string or bytes-like object, got 'NoneType'


In [ ]:
pattern = "([A-Za-z]+\\s*[A-Za-z\\s]+)\\s*\\([A-Za-z\\s]+\\)\\s*Total\\s*Experience\\s*([\\d.\\+]+\\s*[Yy]ears?).+?Since\\s*([A-Za-z]+\\s*[\\d,\\s]+)"
for fund, content in final_text.items():
    check = 'before.fund_manager'
    if check in content:
        text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[check]).strip()
        print(text)
        match = re.findall(pattern,text, re.IGNORECASE)
        print(match)

In [32]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\13_31-Oct-25_FS.json
